In [ ]:
import math
from dataclasses import replace

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

from pathtracer import Renderer, lattice_preview

from spectral import lattice_geometry, row_twiddle, fft, ifft, bin_freq, alias_radius_sq
from utils import device_mapper, host

In [ ]:
DEVICE = device_mapper()

In [ ]:
scene = Renderer("scenes/cornell.json")
rendercam = replace(scene.default_camera)
halves = torch.from_numpy(np.stack([scene.render(camera=rendercam, aovs=("luminance",), width=64, height=32, samples=scene.default_samples // 2, seed=seed, show_sky=True)["luminance"][..., 0] for seed in (1, 2)])).to(DEVICE)

In [ ]:
plt.imshow(lattice_preview(host(halves.mean(0)[..., None]), rendercam, scale=8))
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
# luminance > log > noise-aware whitening > G2 quadrature energy > divisive normalisation

In [ ]:
# Compression: log brightness like photoreceptors; the measured noise level offsets black so it stays finite.
dark = ((halves[0] - halves[1]) ** 2 / 4).mean().sqrt()
log_x = torch.log(halves + dark)

In [ ]:
# Hex-grid frequency geometry: lowest is the coarsest detail the image holds, limit the finest resolved equally in all directions.
row_pitch, row_shift = lattice_geometry(rendercam)
twiddle = row_twiddle(halves.shape[-2:], row_shift, DEVICE)
fy, fx = bin_freq(halves.shape[-2:], row_pitch, DEVICE)
radius_sq, true_fy, true_fx = alias_radius_sq(fy, fx, row_pitch, row_shift)
radius = radius_sq.sqrt()
ac = radius > 0
lowest = 1 / min(halves.shape[-1], halves.shape[-2] * row_pitch)
limit = min(math.hypot(i, (j - i * row_shift) / row_pitch) for i, j in ((0, 1), (1, 0), (1, 1))) / 2
ring = (radius / lowest).long()
count = torch.zeros(int(ring.max()) + 1, device=DEVICE).index_add_(0, ring.flatten(), ac.flatten().float())
ring_mean = lambda v: torch.zeros_like(count).index_add_(0, ring.flatten(), torch.where(ac, v, 0).flatten()) / count

In [ ]:
# Whitening: boosts detail levels where signal beats noise, cuts the rest; two noisy copies set the split, as in retina and fly eye.
first, other = fft(log_x, twiddle)
signal = ring_mean((first * other.conj()).real).clamp(min=0)
noise = ring_mean((first - other).abs() ** 2 / 4)
gain = signal.sqrt() / (signal + noise)
transfer = torch.where(ac, gain[ring], 0)
whitened = (first + other) / 2 * transfer

In [ ]:
# # G2 (cos²) and its Riesz quadrature H2 (cos³) share one radial profile, so hypot is phase-invariant edge energy at every theta
# n = 36
# theta = torch.pi * torch.arange(n, device=DEVICE)[:, None, None] / n
# peak = 1 / 4  # cycles per pixel: Sobel's peak, lambda = 4 px
# radial = radius ** 2 * torch.exp(-(radius / peak) ** 2)
# cos = torch.cos(torch.atan2(true_fy, true_fx) - theta)
# g2 = ifft(whitened * radial * cos ** 2, twiddle).real
# h2 = ifft(whitened * radial * cos ** 3, twiddle).imag
# edges = torch.hypot(g2, h2).amax(0)

# plt.figure(figsize=(10, 5))
# plt.imshow(lattice_preview(host(edges[..., None]), rendercam)[..., 0], cmap="gray")
# plt.title("G2/H2")
# plt.xticks([]), plt.yticks([])
# plt.tight_layout()

In [ ]:
# Edge cells: 0°/60°/120° line+step pairs at the finest reliable scale (simple cells); energy ignores edge position (complex cells).
# G2 and its exact Hilbert pair: one-sided cos² gives even (real) and odd (imag) simple cells; |.|² is the complex cell
orientations = 3  # G2 steers exactly from 3 (Freeman & Adelson 1991)
theta = torch.pi * torch.arange(orientations, device=DEVICE)[:, None, None] / orientations
peak = limit / 2  # one octave below the isotropic band limit; Sobel's 1/4 cycle per pixel on the square lattice
radial = torch.where(radius < limit, radius ** 2 * torch.exp(-(radius / peak) ** 2), 0)
cos = torch.cos(torch.atan2(true_fy, true_fx) - theta)
energy = ifft(whitened * radial * 2 * cos.clamp(min=0) ** 2, twiddle).abs() ** 2

In [ ]:
# Normalisation: detector / (local total + scene median) gives contrast relative to the scene; gain control as in cortex and fly lamina.
# Heeger (1992) normalisation over orientation; semi-saturation at the median pooled energy, Laughlin's (1981) infomax match
pool = energy.sum(0)
response = energy / (pool.median() + pool)

In [ ]:
# Detector votes -> direction + strength per pixel; shown as hue at full strength, strength in grey, and hue over the scene.
# Energy goes as cos⁴ to theta_k, so its square root is cos² and the doubled-angle sum decodes orientation exactly
vector = (response.sqrt() * torch.exp(2j * theta)).sum(0)
hue = (torch.atan2(vector.imag, vector.real) / (2 * torch.pi)) % 1  # edge normal, clockwise from horizontal as rows run down
strength = response.sum(0)
tone = (log_x.mean(0) - log_x.min()) / (log_x.max() - log_x.min())
zeros, ones = torch.zeros_like(strength), torch.ones_like(strength)
hsv = torch.stack([torch.stack(c, -1) for c in ((hue, ones, strength), (hue, zeros, strength), (hue, strength, tone))])
titles = ("G2 energy, normalised: hue = orientation, value = contrast response", "Edge strength", "Edge orientation over the scene, saturation = strength")

fig, ax = plt.subplots(3, 1, figsize=(10, 15), layout="constrained")
for a, panel, title in zip(ax, hsv_to_rgb(host(hsv)), titles):
    a.imshow(lattice_preview(panel, rendercam))
    a.set_title(title)
    a.set_xticks([]), a.set_yticks([])

In [ ]:
# Saliency: where to look. Each pixel scores how rare its strongest edge is in this scene; the brightest spot is the gaze target.
# V1 saliency (Zhaoping 2002) = max over cells of surprise; Gaussian-null quadrature energy is exponential: -log P(E ≥ e) = e / mean
saliency = (energy / energy.mean((-2, -1), keepdim=True)).amax(0)

plt.figure(figsize=(10, 5))
plt.imshow(lattice_preview(host(saliency[..., None]), rendercam)[..., 0])
plt.title("V1 saliency")
plt.xticks([]), plt.yticks([])
plt.tight_layout()